# Práctica 02: LCD 16x2 y Keypad 4x4 con STM32 NUCLEO-C031C6 en Wokwi

Abra esta práctica en Binder para seguir la explicación paso a paso en notebook.


---


## 1. ¿En qué consiste la práctica?

Esta práctica tiene como objetivo integrar dos periféricos digitales comunes:

- Un **LCD 16x2 compatible con HD44780** en **modo 4 bits**
- Un **teclado matricial 4x4 de membrana**

Ambos se conectan a la tarjeta **STM32 NUCLEO-C031C6** dentro del simulador **Wokwi**.  
La aplicación final detecta la tecla presionada y la muestra en la pantalla LCD.

### Objetivo funcional
Cada vez que el usuario pulse una tecla del keypad:
- el sistema la detecta,
- identifica qué carácter corresponde,
- y muestra en el LCD **solo la última tecla pulsada**.

### Resultado esperado
- Primera línea del LCD: texto fijo, por ejemplo `Tecla Presionada`
- Segunda línea del LCD: la tecla más reciente, por ejemplo `5`, `A`, `#`, etc.


## 2. Elementos utilizados

### Hardware simulado en Wokwi
- **STM32 NUCLEO-C031C6**
- **LCD 16x2**
- **wokwi-membrane-keypad** de 4x4

### Software
- **Wokwi** para simulación
- **STM32CubeMX** para la configuración de pines
- **HAL de STM32** para el control de GPIO y retardos


## 3. Librerías usadas

### `main.h`
Archivo generado por STM32CubeMX.  
Contiene definiciones de pines, puertos, prototipos y configuración general del proyecto.

### `stm32c0xx_hal.h`
Cabecera principal de la librería HAL de STM32.  
Permite acceder a funciones como:
- `HAL_GPIO_WritePin()`
- `HAL_GPIO_ReadPin()`
- `HAL_Delay()`

### `lcd16x2_hal.h` y `lcd16x2_hal.c`
Librería creada para esta práctica.  
Su función es controlar el LCD 16x2 en modo 4 bits.

Se encarga de:
- enviar comandos al LCD,
- enviar caracteres,
- posicionar el cursor,
- limpiar pantalla,
- inicializar el display.

### `keypad4x4_hal.h` y `keypad4x4_hal.c`
Librería creada para esta práctica.  
Su función es escanear el teclado matricial 4x4.

Se encarga de:
- activar una fila a la vez,
- leer las columnas,
- identificar la tecla pulsada,
- devolver el carácter correspondiente.


## 4. Explicación breve del LCD en modo 4 bits

El LCD 16x2 puede trabajar en dos modos:
- **8 bits**
- **4 bits**

En esta práctica se usa **modo 4 bits** para ahorrar pines del microcontrolador.  
Esto significa que cada byte se envía en dos partes:
1. nibble alto
2. nibble bajo

### Líneas principales del LCD usadas
- `RS`: selecciona comando o dato
- `E`: habilita la transferencia
- `D4-D7`: líneas de datos
- `RW`: se conecta a GND para solo escritura


## 5. Explicación breve del keypad 4x4

El teclado 4x4 está organizado como una matriz de:
- **4 filas**
- **4 columnas**

Cada tecla une una fila con una columna.  
Para detectar una tecla:
1. se coloca una fila en estado bajo,
2. se dejan las demás en alto,
3. se leen las columnas,
4. si una columna baja, se identifica la tecla en la intersección.

### Ventaja
Permite leer **16 teclas usando solo 8 pines**.


## 6. Conexiones del circuito

### LCD 16x2
- `RS  -> PB4`
- `E   -> PB5`
- `D4  -> PA0`
- `D5  -> PA1`
- `D6  -> PA4`
- `D7  -> PA7`
- `RW  -> GND`
- `VSS -> GND`
- `VDD -> 5V`
- `V0  -> GND` *(o a potenciómetro si se desea ajustar contraste)*
- `A   -> 5V`
- `K   -> GND`

### Keypad 4x4
- `R1 -> PA8`
- `R2 -> PA9`
- `R3 -> PA10`
- `R4 -> PA11`
- `C1 -> PB0`
- `C2 -> PB1`
- `C3 -> PB6`
- `C4 -> PB7`


## 7. I/O Map

| Elemento | Señal | Puerto/PIN STM32 | Dirección |
|---|---|---|---|
| LCD | RS | PB4 | Salida |
| LCD | E  | PB5 | Salida |
| LCD | D4 | PA0 | Salida |
| LCD | D5 | PA1 | Salida |
| LCD | D6 | PA4 | Salida |
| LCD | D7 | PA7 | Salida |
| Keypad | R1 | PA8 | Salida |
| Keypad | R2 | PA9 | Salida |
| Keypad | R3 | PA10 | Salida |
| Keypad | R4 | PA11 | Salida |
| Keypad | C1 | PB0 | Entrada Pull-Up |
| Keypad | C2 | PB1 | Entrada Pull-Up |
| Keypad | C3 | PB6 | Entrada Pull-Up |
| Keypad | C4 | PB7 | Entrada Pull-Up |


## 8. Funciones implementadas en la librería del LCD

### `LCD_Init()`
Inicializa el LCD en modo 4 bits.  
Envía la secuencia requerida por el controlador HD44780 y configura:
- interfaz de 4 bits,
- 2 líneas,
- display encendido,
- cursor apagado.

### `LCD_SendCommand()`
Envía un comando al LCD, por ejemplo:
- limpiar pantalla,
- mover cursor,
- configurar display.

### `LCD_SendChar()`
Envía un solo carácter al LCD.

### `LCD_SendString()`
Envía una cadena completa carácter por carácter.

### `LCD_SetCursor()`
Coloca el cursor en una posición específica:
- fila 0 o 1
- columna 0 a 15

### `LCD_Clear()`
Limpia la pantalla y regresa el cursor al inicio.


## 9. Funciones implementadas en la librería del keypad

### `Keypad_Init()`
Coloca las filas en estado alto al inicio.

### `Keypad_GetKey()`
Realiza el escaneo del teclado:
1. pone una fila en bajo,
2. revisa las columnas,
3. si detecta una pulsación, aplica un pequeño debounce,
4. espera la liberación de la tecla,
5. regresa el carácter asociado.

### Mapa de teclas usado
```c
{
    {'1','2','3','A'},
    {'4','5','6','B'},
    {'7','8','9','C'},
    {'*','0','#','D'}
}
```


## 10. Flujo general de funcionamiento

1. Se inicializa HAL.
2. Se configura el reloj del sistema.
3. Se inicializan los GPIO con CubeMX.
4. Se construyen las estructuras del LCD y del keypad.
5. Se inicializa el LCD.
6. Se inicializa el keypad.
7. En el ciclo principal:
   - se lee una tecla,
   - si hay una pulsación válida,
   - se actualiza el LCD con la última tecla detectada.


## 11. Recomendaciones de implementación

- Configurar las **filas del keypad como salida**
- Configurar las **columnas como entrada con pull-up**
- Mantener `RW` del LCD a tierra
- Añadir pequeños retardos en la inicialización del LCD
- Evitar conflictos de nombres entre macros de CubeMX y campos de estructuras

### Ejemplo de buena práctica
No usar nombres de campos como:
- `RS_Pin`
- `EN_Pin`
- `D4_Pin`

porque CubeMX ya define esos nombres en `main.h`.  
Es preferible usar:
- `rs_pin`
- `en_pin`
- `d4_pin`


## 12. Resultado esperado de la práctica

Al ejecutar la simulación:
- el LCD muestra un mensaje fijo en la primera línea,
- al pulsar una tecla del keypad, aparece en la segunda línea,
- cuando se pulsa otra tecla, la anterior se reemplaza,
- siempre se visualiza solo la **Tecla Presionada**.


## 13. Conclusión

Esta práctica permite reforzar varios conceptos importantes:
- manejo de GPIO con HAL,
- interfaz paralela en 4 bits,
- escaneo matricial de teclado,
- organización de código mediante librerías propias,
- documentación técnica de una práctica para repositorio.

Además, sirve como base para futuras prácticas como:
- captura de PIN,
- menús en LCD,
- control de acceso,
- calculadora básica,
- interfaz usuario-microcontrolador.
